# Tactical System Fit Score Multi-Label Classification

In [4]:
# Load the positional datasets and archetype classification data
import pandas as pd
import numpy as np
from pathlib import Path

root = Path('/Users/mukikrishnan/Desktop/Interactive Soccer Dashboard')

gk_file = root / 'Positional Stats/Goalkeeping/Sorted Data/GoalkeepingSortedData.csv'
def_file = root / 'Positional Stats/Defending/Sorted Data/DefendingSortedData.csv'
mid_file = root / 'Positional Stats/Midfielding/Sorted Data/MidfieldingSortedData.csv'
fwd_file = root / 'Positional Stats/Attacking/Sorted Data/ForwardsSortedData.csv'
archetype_file = root / 'Archetype Classification/CombinedArchetypes.csv'

GK = pd.read_csv(gk_file)
DEF = pd.read_csv(def_file)
MID = pd.read_csv(mid_file)
FWD = pd.read_csv(fwd_file)
ARCH = pd.read_csv(archetype_file)

for df in [GK, DEF, MID, FWD, ARCH]:
    df.columns = [c.strip() for c in df.columns]
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)
    if 'Player' in df.columns:
        df['Player'] = df['Player'].astype(str).str.strip()

ARCH['Ideal Archetype'] = ARCH['Ideal Archetype'].astype(str).str.strip()
GK['source_position_group'] = 'GK'
DEF['source_position_group'] = 'DEF'
MID['source_position_group'] = 'MID'
FWD['source_position_group'] = 'FWD'

# Deduplicate ARCH so we have a single archetype per player before merging
# Inspect duplicates (kept for debugging if needed)
dupes = ARCH[ARCH.duplicated('Player', keep=False)].sort_values(['Player', 'Season'] if 'Season' in ARCH.columns else ['Player'])
if not dupes.empty:
    # Prefer the most recent season if available, otherwise take the modal archetype
    if 'Season' in ARCH.columns:
        ARCH_unique = ARCH.sort_values('Season').drop_duplicates('Player', keep='last')[['Player', 'Ideal Archetype']]
    else:
        ARCH_unique = (
            ARCH.groupby('Player', as_index=False)['Ideal Archetype']
            .agg(lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0])
        )
else:
    ARCH_unique = ARCH[['Player', 'Ideal Archetype']].drop_duplicates('Player')

# Merge archetype labels into each positional dataset using the deduped table
GK = GK.merge(ARCH_unique, on='Player', how='left', validate='one_to_one')
DEF = DEF.merge(ARCH_unique, on='Player', how='left', validate='one_to_one')
MID = MID.merge(ARCH_unique, on='Player', how='left', validate='one_to_one')
FWD = FWD.merge(ARCH_unique, on='Player', how='left', validate='one_to_one')

all_players = pd.concat([GK, DEF, MID, FWD], ignore_index=True, sort=False)
all_players[['Player', 'Pos', 'source_position_group', 'Ideal Archetype']].head(10)


,Player,Pos,source_position_group,Ideal Archetype
0,Alisson,GK,GK,Modern GK
1,Alphonse Areola,GK,GK,Classic GK
2,Simone Aresti,GK,GK,Modern GK
3,Noah Atubolu,GK,GK,Classic GK
4,Oliver Baumann,GK,GK,Classic GK
5,Alaa Bellaarouch,GK,GK,Modern GK
6,Etrit Berisha,GK,GK,Modern GK
7,Marco Bizot,GK,GK,Classic GK
8,Janis Blaswich,GK,GK,Modern GK
9,Marcin Bułka,GK,GK,Sweeper Keeper


In [5]:
# Define tactical systems with expected archetypes and compute support scores
SYSTEM_ARCHETYPES = {
    'Tiki Taka Possession': {
        'Modern GK', 'Ball Playing Center Back', 'Central Defender', 'Full Back',
        'Wing Back', 'Deep Lying Playmaker', 'Regista',
        'Advanced Playmaker', 'Inside Forward', 'False Nine'
    },
    'Gegenpressing Counter Attack': {
        'Sweeper Keeper', 'Central Defender', 'Libero', 'Wing Back',
        'Anchor Man', 'Ball Winning Midfielder', 'Box-to-Box', 'Inverted Winger',
        'Second Striker', 'False Nine', 'Complete Forward'
    },
    'Low Block Defensive': {
        'Classic GK', 'Central Defender', 'Classic Defender', 'Full Back',
        'Anchor Man', 'Ball Winning Midfielder', 'Classic CM',
        'Wide Midfielder', 'Classic Winger', 'Complete Forward'
    },
    'Wing Play': {
        'Classic GK', 'Libero', 'Full Back', 'Inverted Wing Back',
        'Deep Lying Playmaker', 'Mezzala', 'Segundo Volante',
        'Classic Winger', 'Complete Forward', 'Inverted Winger'
    }
}

SYSTEM_POSITION_GROUPS = {
    'Tiki Taka Possession': {'GK', 'DEF', 'MID', 'FWD'},
    'Gegenpressing Counter Attack': {'GK', 'DEF', 'MID', 'FWD'},
    'Low Block Defensive': {'GK', 'DEF', 'MID', 'FWD'},
    'Wing Play': {'GK', 'DEF', 'MID', 'FWD'}
}

# Standardize text for archetype lookups
all_players['Ideal Archetype'] = all_players['Ideal Archetype'].astype(str).str.strip()

# Build normalized stats for each position group
position_normalizers = {}

def normalize_series(series):
    values = pd.to_numeric(series, errors='coerce')
    if values.isna().all():
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - values.min()) / (values.max() - values.min() + 1e-9)

# Feature sets for different tactical profiles
PROFILE_FEATURES = {
    'GK_modern': ['Cmp%', 'AvgDist', 'Launch%', 'Stp%'],
    'DEF_ball_playing': ['Pass Cmp%', 'PrgDist', 'KP', 'Cmp%'],
    'DEF_defensive': ['Tkl+Int', 'Clr', 'Blocks', 'Int'],
    'MID_playmaking': ['KP', 'PPA', 'PrgP', 'xA'],
    'MID_defensive': ['Tkl', 'Int', 'Def 3rd', 'Tkl+Int'],
    'FWD_creative': ['Gls', 'Ast', 'G+A', 'SoT%'],
    'FWD_wide': ['CrsPA', 'Touches', 'KP', 'Gls']
}

for group_name, group_df in [('GK', GK), ('DEF', DEF), ('MID', MID), ('FWD', FWD)]:
    # Use Player names as the index so lookups work after concatenation
    players_index = group_df['Player'].astype(str).str.strip()
    norm_df = pd.DataFrame(index=players_index)
    for feature in set(sum(PROFILE_FEATURES.values(), [])):
        if feature in group_df.columns:
            norm_series = normalize_series(group_df[feature])
            # align the normalized series to the Player index
            norm_series.index = players_index
            norm_df[feature] = norm_series
    position_normalizers[group_name] = norm_df

# Mapping of archetypes to support profiles
ARCHETYPE_SUPPORT = {
    'Modern GK': ('GK_modern', 1.0),
    'Sweeper Keeper': ('GK_modern', 0.8),
    'Classic GK': ('GK_modern', 0.6),
    'Ball Playing Center Back': ('DEF_ball_playing', 1.0),
    'Central Defender': ('DEF_defensive', 0.8),
    'Classic Defender': ('DEF_defensive', 0.9),
    'Libero': ('DEF_ball_playing', 1.0),
    'Classic Full Back': ('DEF_ball_playing', 0.7),
    'Wing Back': ('DEF_ball_playing', 0.8),
    'Inverted Wing Back': ('DEF_ball_playing', 0.9),
    'Anchor Man': ('MID_defensive', 1.0),
    'Ball Winning Midfielder': ('MID_defensive', 1.0),
    'Box-to-Box': ('MID_playmaking', 0.7),
    'Deep Lying Playmaker': ('MID_playmaking', 1.0),
    'Regista': ('MID_playmaking', 1.0),
    'Advanced Playmaker': ('MID_playmaking', 1.0),
    'Mezzala': ('MID_playmaking', 0.9),
    'Segundo Volante': ('MID_defensive', 0.85),
    'Classic CM': ('MID_playmaking', 0.7),
    'Wide Midfielder': ('MID_playmaking', 0.6),
    'Wide Playmaker': ('MID_playmaking', 0.8),
    'Classic Winger': ('FWD_wide', 1.0),
    'Inverted Winger': ('FWD_creative', 0.95),
    'Inside Forward': ('FWD_creative', 1.0),
    'Second Striker': ('FWD_creative', 0.9),
    'Complete Forward': ('FWD_creative', 0.95),
    'False Nine': ('FWD_creative', 1.0),
    'Target Man': ('FWD_creative', 0.7),
    'Poacher': ('FWD_creative', 0.8)
}

# helper to compute stat support for a row

def row_feature_support(row, group_features):
    group = row.get('source_position_group')
    if pd.isna(group) or group not in position_normalizers:
        return 0.0
    support_df = position_normalizers[group]
    player = str(row.get('Player', '')).strip()
    values = []
    for feature in group_features:
        if feature in support_df.columns and player in support_df.index:
            values.append(support_df.at[player, feature])
    return np.nanmean(values) if values else 0.0


def compute_system_fit(row, system_name):
    archetype = str(row.get('Ideal Archetype', '')).strip()
    archetype_match = 1.0 if archetype in SYSTEM_ARCHETYPES[system_name] else 0.0
    position_match = 1.0 if row['source_position_group'] in SYSTEM_POSITION_GROUPS[system_name] else 0.0

    stat_support = 0.0
    if archetype in ARCHETYPE_SUPPORT:
        profile, archetype_weight = ARCHETYPE_SUPPORT[archetype]
        features = PROFILE_FEATURES.get(profile, [])
        stat_support = row_feature_support(row, features) * archetype_weight
    else:
        group = row['source_position_group']
        if group == 'GK':
            stat_support = row_feature_support(row, PROFILE_FEATURES['GK_modern']) * 0.6
        elif group == 'DEF':
            stat_support = row_feature_support(row, PROFILE_FEATURES['DEF_defensive']) * 0.6
        elif group == 'MID':
            stat_support = row_feature_support(row, PROFILE_FEATURES['MID_playmaking']) * 0.6
        elif group == 'FWD':
            stat_support = row_feature_support(row, PROFILE_FEATURES['FWD_creative']) * 0.6

    # Score is a weighted combination of archetype match, positional fit, and stat support.
    return 0.45 * archetype_match + 0.20 * position_match + 0.35 * stat_support

for system in SYSTEM_ARCHETYPES.keys():
    score_column = f'Fit_{system.replace(" ", "_").replace("-", "_").replace("/", "_")}'
    all_players[score_column] = all_players.apply(lambda row: compute_system_fit(row, system), axis=1)

# Show the top-fit players for each system
for system in SYSTEM_ARCHETYPES.keys():
    score_column = f'Fit_{system.replace(" ", "_").replace("-", "_").replace("/", "_")}'
    display_df = all_players.sort_values(score_column, ascending=False)[
        ['Player', 'Pos', 'source_position_group', 'Ideal Archetype', score_column]
    ].head(15)
    print(f'\nTop 15 players for {system}:')
    display(display_df)


Top 15 players for Tiki Taka Possession:


,Player,Pos,source_position_group,Ideal Archetype,Fit_Tiki_Taka_Possession
2141,Khvicha Kvaratskhelia,LW,FWD,Inside Forward,0.908920
57,Unai Marrero,GK,GK,Modern GK,0.882287
2140,Dejan Kulusevski,RW,FWD,Inside Forward,0.880911
67,Juan Musso,GK,GK,Modern GK,0.865841
2158,Rafael Leão,LW,FWD,Inside Forward,0.864796
2095,Junya Ito,RW,FWD,Inside Forward,0.859580
2194,Kylian Mbappé,LW,FWD,Inside Forward,0.858582
31,Aitor Fernández,GK,GK,Modern GK,0.857900
2139,Mohammed Kudus,RW,FWD,Inside Forward,0.842029
80,Jonas Omlin,GK,GK,Modern GK,0.835562



Top 15 players for Gegenpressing Counter Attack:


,Player,Pos,source_position_group,Ideal Archetype,Fit_Gegenpressing_Counter_Attack
485,Jan Paul van Hecke,CB,DEF,Libero,0.990265
171,Waldemar Anton,CB,DEF,Libero,0.860715
140,Emmanuel Agbadou,CB,DEF,Libero,0.854253
66,Arijanet Muric,GK,GK,Sweeper Keeper,0.851986
149,Manuel Akanji,CB,DEF,Libero,0.851836
247,Lilian Brassier,CB,DEF,Libero,0.846037
90,David Raya,GK,GK,Sweeper Keeper,0.845558
231,Daley Blind,CB,DEF,Libero,0.844846
294,Brendan Chardonnet,CB,DEF,Libero,0.841971
130,Yunis Abdelhamid,CB,DEF,Libero,0.840906



Top 15 players for Low Block Defensive:


,Player,Pos,source_position_group,Ideal Archetype,Fit_Low_Block_Defensive
1014,Alessandro Vogliacco,CB,DEF,Central Defender,0.811277
1055,Christoph Zimmermann,CB,DEF,Central Defender,0.806614
767,Adam Obert,CB,DEF,Central Defender,0.805067
1028,Mateusz Wieteska,CB,DEF,Central Defender,0.804064
675,Lisandro Martínez,CB,DEF,Central Defender,0.798433
712,Éder Militão,CB,DEF,Central Defender,0.798125
1053,Nathan Zeze,CB,DEF,Central Defender,0.796297
658,Kostas Manolas,CB,DEF,Central Defender,0.795200
820,Kamil Piątkowski,CB,DEF,Central Defender,0.794207
1011,Mattia Viti,CB,DEF,Central Defender,0.791954



Top 15 players for Wing Play:


,Player,Pos,source_position_group,Ideal Archetype,Fit_Wing_Play
485,Jan Paul van Hecke,CB,DEF,Libero,0.990265
496,Theo Hernández,LB,DEF,Inverted Wing Back,0.909014
593,Kenny Lala,RB,DEF,Inverted Wing Back,0.874575
624,Bradley Locko,LB,DEF,Inverted Wing Back,0.871622
171,Waldemar Anton,CB,DEF,Libero,0.860715
140,Emmanuel Agbadou,CB,DEF,Libero,0.854253
149,Manuel Akanji,CB,DEF,Libero,0.851836
516,Ismaily,LB,DEF,Inverted Wing Back,0.848783
247,Lilian Brassier,CB,DEF,Libero,0.846037
231,Daley Blind,CB,DEF,Libero,0.844846


In [6]:
# Save the scored dataset to CSV in the 'Tactical System Fit Score' folder
from pathlib import Path
from datetime import datetime

out_dir = root / 'Tactical System Fit Score'
out_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
out_file = out_dir / f'all_players_system_fit_scores_{timestamp}.csv'
all_players.to_csv(out_file, index=False)
print(f"Saved scored dataset to: {out_file}")

Saved scored dataset to: /Users/mukikrishnan/Desktop/Interactive Soccer Dashboard/Tactical System Fit Score/all_players_system_fit_scores_20260620_135759.csv
